# 04 - Friction Measures Home (v4, generated 2026-10-13)

Fabric folder: Dev_CGP > Notebooks > 03 - EADM > Collaboration. Run order: 01 - Policy Renewals 30d, 02 - Friction Frame, then this notebook (03 Car first,
which builds the shared call table stg.fe3m_calls_x).

The Home measures of the friction extract (spec v3.3) on the frame's episodes: one row per Home episode in
`stg.fe3m_home` with all 104 Home fields. Same rules as 03 - Friction Measures Car:
flags 1 / 0 on their own episode type, blank elsewhere; blank before the reliable-from date or when an eligible
episode meets neither rule; branches blank while open on DataDate; a long call = queue + ring time >=
`WaitThresholdSeconds` or no agent answered. Writes `stg.fe3m_home*` only.

Agreed for Home (2026-10-13): built where the spec asked for confirmation but the stream has the
events (F3, F4, R1, R3, C4; R5 from HR0.F2, the held letter); left blank: F5, C5 (no source) and the loan-setup
branch (ignore). R4: the no-price event carries no quote reference, so it is read per episode. C1 resolution is
Unknown (spec). v4: R6 counts HR1.E1.F1 up to the episode end, not only to the due date (about half fall after it;
validation pack T32).
Branches are entered on their own spine (happy-path steps from the flow sheets), not on their friction or outcome steps.
v2: F2 - HA4b is recorded on online Home sales too, so an online purchase (HA4a) decides when both are present.


In [1]:
-- Parameters. Reliable-from dates apply to EpisodeStart.
--   RF_Calls: Genesys queues complete from Nov 2022 (F1, R2, R3, C1, C3, C4).
--   RF_F2, RF_F3: Home online quotes (HA1 / HA3 / HA3.F1 / HA4a) from late Feb 2025 (spec: Feb 2025).
--   RF_F4: HA7.B1.4 from Feb 2024 (Q9: events exist, built as agreed 2026-10-13).
--   RF_R1: HR2.F1 failed logins from mid May 2025 (DE1: built as agreed). RF_R4, RF_Requote: Home re-quotes from late
--   Feb 2025 (spec: Feb 2025 or May 2026, DE4). RF_R5: HR0.F2 held letter (Vijay) from 2021.
--   RF_R6: HR1.E1.F1 from Apr 2024. RF_FyDoc: HA6 from Feb 2024. RF_RnDoc, RF_ChangeDoc: document intake from Oct
--   2023 (as Car). RF_Monthly: HR4c from 2021.
Set WaitThresholdSeconds = 600 ;
Set PreSaleLookbackDays = 30 ;
Set RF_Calls = '2022-11-01' ;
Set RF_F2 = '2025-03-01' ;
Set RF_F3 = '2025-03-01' ;
Set RF_F4 = '2024-03-01' ;
Set RF_R1 = '2025-06-01' ;
Set RF_R4 = '2025-03-01' ;
Set RF_R5 = '2021-07-01' ;
Set RF_R6 = '2024-05-01' ;
Set RF_FyDoc = '2024-03-01' ;
Set RF_Requote = '2025-03-01' ;
Set RF_RnDoc = '2023-10-01' ;
Set RF_Monthly = '2021-07-01' ;
Set RF_ChangeDoc = '2023-10-01' ;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 16, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

... (6 outputs left out)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

In [2]:
-- Home episodes from the frame, with the window end used by "before DecisionDate" rules and an open flag.
CREATE OR REPLACE TABLE stg.fe3m_home_ep AS
SELECT e.*,
       least(e.EpisodeEnd, COALESCE(e.DecisionDate, e.EpisodeEnd)) AS win_end,
       CASE WHEN e.EpisodeEnd > e.DataDate THEN 1 ELSE 0 END          AS is_open
FROM   stg.fe3_episode e
WHERE  e.Product = 'Home'
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
-- Home events used by the measures (distinct: identical copies count once).
CREATE OR REPLACE TABLE stg.fe3m_home_events AS
SELECT DISTINCT EventTypeId, EventDate, TRIM(SourcePolicyReference) AS pc, SourceQuoteReference AS qr,
       CASE WHEN SCV_Customer_Key > 0 THEN SCV_Customer_Key END AS scv
FROM   ods.episodeeventsnumberedfinal
WHERE  ProductType = 'home' AND EventDate >= '2022-01-01' AND EventDate <= current_date()
AND   (EventTypeId IN ('HA1', 'HA3', 'HA3.F1', 'HA4a', 'HA4b', 'HA4.F1', 'HA7', 'HA7.B1.4', 'HA6', 'HR2a', 'HR2.F1', 'HR0.F2', 'HR1a', 'HR1b', 'HR1.E1.F1', 'HR4', 'HR4c', 'HM3')
       OR EventTypeId LIKE 'HA6.B1%' OR EventTypeId LIKE 'HR2.B1%' OR EventTypeId LIKE 'HR4.B2%' OR EventTypeId LIKE 'HM4.B1%' )
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
-- Call measures: per episode, the calls of a line inside the measure's window (calls from 03: stg.fe3m_calls_x).
CREATE OR REPLACE TABLE stg.fe3m_home_callagg AS
WITH w AS (
    SELECT e.EpisodeID, 'F1' AS m, c.*
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'sales' AND c.queue_product = 'Home'
           AND c.call_date BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, 'R2', c.*
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Home'
           AND c.call_date BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT e.EpisodeID, 'C1', c.*
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'change' AND c.queue_product = 'Home'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
    UNION ALL
    SELECT e.EpisodeID, 'C3', c.*
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'document'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Document request'
),
w2 AS (
    SELECT w.*, MIN(CASE WHEN is_long = 1 THEN call_ts END) OVER (PARTITION BY EpisodeID, m) AS first_long
    FROM   w
)
SELECT EpisodeID, m, COUNT(*) AS n, SUM(connected) AS n_conn, SUM(1 - connected) AS n_aband,
       CAST(ROUND(MAX(wait_s)) AS INT) AS max_wait, SUM(is_long) AS n_long, MAX(first_long) AS first_long,
       MAX(CASE WHEN connected = 1 AND call_ts >= first_long THEN 1 ELSE 0 END) AS resolved
FROM   w2 GROUP BY EpisodeID, m
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
-- R3: first online session (login) per renewal episode, then whether a renewal-line call follows (date level).
CREATE OR REPLACE TABLE stg.fe3m_home_r3 AS
WITH l AS (
    SELECT e.EpisodeID, MIN(v.EventDate) AS first_login
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_home_events v
           ON v.scv = e.scv_id AND v.EventTypeId = 'HR2a' AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    GROUP  BY e.EpisodeID
)
SELECT l.EpisodeID, l.first_login,
       MAX(CASE WHEN c.conversationId IS NOT NULL THEN 1 ELSE 0 END) AS called_after
FROM   l JOIN stg.fe3m_home_ep e ON e.EpisodeID = l.EpisodeID
LEFT JOIN stg.fe3m_calls_x c
       ON c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Home'
       AND c.call_date >= l.first_login AND c.call_date <= e.win_end
GROUP  BY l.EpisodeID, l.first_login
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
-- Renewal episodes under each code their events can carry: the policy and, if different, its renewed code.
CREATE OR REPLACE TABLE stg.fe3m_home_rcodes AS
SELECT EpisodeID, PolicyRef AS code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_home_ep WHERE EpisodeType = 'Renewal'
UNION
SELECT EpisodeID, new_policy_code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_home_ep
WHERE  EpisodeType = 'Renewal' AND new_policy_code IS NOT NULL AND new_policy_code <> PolicyRef
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
-- Event aggregates per episode: quote-keyed first-year events by customer in the 30 days before the sale; policy-
-- keyed events by the policy inside the episode (renewal events also under the renewed code); logins by customer.
CREATE OR REPLACE TABLE stg.fe3m_home_evagg AS
WITH x AS (
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_home_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('HA1', 'HA3', 'HA3.F1', 'HA4a')
           AND v.EventDate BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_home_events v
           ON  v.pc = e.PolicyRef
           AND (v.EventTypeId IN ('HA4b', 'HA4.F1', 'HA6', 'HA7', 'HA7.B1.4') OR v.EventTypeId LIKE 'HA6.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_home_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('HR2a', 'HR2.F1')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT r.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_home_rcodes r JOIN stg.fe3m_home_events v
           ON  v.pc = r.code
           AND (v.EventTypeId IN ('HR0.F2', 'HR1a', 'HR1b', 'HR1.E1.F1', 'HR4', 'HR4c') OR v.EventTypeId LIKE 'HR2.B1%' OR v.EventTypeId LIKE 'HR4.B2%' )
           -- invitations, held letters and payment steps carry derived dates (due - 60 / due date): a wider band
           AND v.EventDate BETWEEN date_sub(r.DueDate, 90) AND date_add(r.EpisodeEnd, 16)
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_home_ep e JOIN stg.fe3m_home_events v
           ON  v.pc = e.PolicyRef AND (v.EventTypeId = 'HM3' OR v.EventTypeId LIKE 'HM4.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
)
SELECT x.EpisodeID,
       MAX(CASE WHEN EventTypeId = 'HA1' THEN 1 ELSE 0 END)                  AS q_online,
       MAX(CASE WHEN EventTypeId = 'HA3' THEN 1 ELSE 0 END)                     AS got_price,
       MAX(CASE WHEN EventTypeId = 'HA3.F1' THEN 1 ELSE 0 END)               AS no_price_online,
       MAX(CASE WHEN EventTypeId = 'HA4a' THEN 1 ELSE 0 END)                 AS bought_online,
       MAX(CASE WHEN EventTypeId IN ('HA4b', 'HA4.F1') THEN 1 ELSE 0 END)          AS bought_by_phone,
       MAX(CASE WHEN EventTypeId = 'HA6' THEN 1 ELSE 0 END)                AS fy_doc_reached,
       SUM(CASE WHEN EventTypeId IN ('HA6.B1', 'HA6.B1.1', 'HA6.B1.2', 'HA6.B1.3') THEN 1 ELSE 0 END) AS fy_doc_n,
       MIN(CASE WHEN EventTypeId = 'HA7' THEN EventDate END)                     AS dispatch_first,
       SUM(CASE WHEN EventTypeId = 'HR2a' THEN 1 ELSE 0 END)                      AS logins_ok,
       SUM(CASE WHEN EventTypeId = 'HR2.F1' THEN 1 ELSE 0 END)                  AS logins_failed,
       MIN(CASE WHEN EventTypeId = 'HR2.F1' THEN EventDate END)                 AS first_failed,
       MAX(CASE WHEN EventTypeId = 'HR2a' THEN EventDate END)                     AS last_ok,
       MAX(CASE WHEN EventTypeId = 'HR0.F2' AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS held,
       MIN(CASE WHEN EventTypeId = 'HR0.F2' AND EventDate <= e.DueDate THEN EventDate END) AS held_first,
       MAX(CASE WHEN EventTypeId IN ('HR1a', 'HR1b') AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS invited,
       MAX(CASE WHEN EventTypeId IN ('HR1a', 'HR1b') AND EventDate <= e.DueDate THEN EventDate END) AS invited_last,
       -- R6: the date-missed event falls after the due date as often as before it: count it to the episode end
       MAX(CASE WHEN EventTypeId = 'HR1.E1.F1' AND EventDate <= e.EpisodeEnd THEN 1 ELSE 0 END) AS date_missed,
       SUM(CASE WHEN EventTypeId LIKE 'HR2.B1%' AND EventDate <= e.win_end THEN 1 ELSE 0 END) AS requote_n,
       SUM(CASE WHEN EventTypeId IN ('HR2.B1', 'HR2.B1.1', 'HR2.B1.2', 'HR2.B1.3')
                AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd THEN 1 ELSE 0 END)   AS requote_win_n,
       COUNT(DISTINCT CASE WHEN EventTypeId = 'HR2.B1.2' AND EventDate <= e.win_end THEN COALESCE(qr, CAST(EventDate AS STRING)) END)
                                                                                            AS rq_with_price,
       SUM(CASE WHEN EventTypeId = 'HR2.B1.2.F1' AND EventDate <= e.win_end THEN 1 ELSE 0 END) AS rq_no_price,
       MIN(CASE WHEN EventTypeId = 'HR2.B1.2.F1' AND EventDate <= e.win_end THEN EventDate END) AS first_no_price,
       MAX(CASE WHEN EventTypeId = 'HR2.B1.2' AND EventDate <= e.win_end THEN EventDate END)    AS last_price,
       MAX(CASE WHEN EventTypeId = 'HR4' THEN 1 ELSE 0 END)                          AS paid,
       SUM(CASE WHEN EventTypeId = 'HR4c' THEN 1 ELSE 0 END)                   AS monthly_n,
       SUM(CASE WHEN EventTypeId IN ('HR4.B2', 'HR4.B2.1', 'HR4.B2.2', 'HR4.B2.3')
                AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd THEN 1 ELSE 0 END)   AS rn_doc_n,
       MAX(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END) AS loan_reached,
       SUM(CASE WHEN EventTypeId IN ('__none__') THEN 1 ELSE 0 END) AS loan_n,
       SUM(CASE WHEN EventTypeId LIKE '__none__%' AND EventDate >= e.SaleDate THEN 1 ELSE 0 END)
                                                                                            AS loan_after_sale_n,
       SUM(CASE WHEN EventTypeId = '__none__' AND EventDate >= e.SaleDate THEN 1 ELSE 0 END)
                                                                                            AS loan_chase_n,
       MAX(CASE WHEN EventTypeId = 'HM3' THEN 1 ELSE 0 END)                AS change_reached,
       SUM(CASE WHEN EventTypeId IN ('HM4.B1', 'HM4.B1.1', 'HM4.B1.2', 'HM4.B1.3') THEN 1 ELSE 0 END) AS change_doc_n
FROM   x JOIN stg.fe3m_home_ep e USING (EpisodeID)
GROUP  BY x.EpisodeID
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
-- F4: chases after the first dispatch; resolved if the documents are dispatched again after the first chase.
CREATE OR REPLACE TABLE stg.fe3m_home_ev2 AS
WITH k AS (
    SELECT a.EpisodeID, e.PolicyRef, e.EpisodeEnd, a.dispatch_first
    FROM   stg.fe3m_home_evagg a JOIN stg.fe3m_home_ep e ON e.EpisodeID = a.EpisodeID
    WHERE  e.EpisodeType = 'First year' AND a.dispatch_first IS NOT NULL
),
ch AS (
    SELECT k.EpisodeID, COUNT(v.EventDate) AS chases, MIN(v.EventDate) AS first_chase
    FROM   k LEFT JOIN stg.fe3m_home_events v
           ON v.pc = k.PolicyRef AND v.EventTypeId = 'HA7.B1.4'
           AND v.EventDate >= k.dispatch_first AND v.EventDate <= k.EpisodeEnd
    GROUP  BY k.EpisodeID
)
SELECT ch.EpisodeID, ch.chases, ch.first_chase,
       MAX(CASE WHEN v.EventDate > ch.first_chase AND v.EventDate <= k.EpisodeEnd THEN 1 ELSE 0 END) AS redispatched
FROM   ch JOIN k ON k.EpisodeID = ch.EpisodeID
LEFT JOIN stg.fe3m_home_events v ON v.pc = k.PolicyRef AND v.EventTypeId = 'HA7'
GROUP  BY ch.EpisodeID, ch.chases, ch.first_chase
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
-- The Home measures: one row per Home episode, all 104 fields in the spec's order.
CREATE OR REPLACE TABLE stg.fe3m_home AS
SELECT e.EpisodeID, e.EpisodeType,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(f1.n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_FirstYear_F1_SalesCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Home_FirstYear_F1_SalesCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND (NOT (COALESCE(f1.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_FirstYear_F1_SalesCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT ((COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN f1.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_FirstYear_F1_SalesCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n END
           AS Home_FirstYear_F1_SalesCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_conn END
           AS Home_FirstYear_F1_SalesCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_aband END
           AS Home_FirstYear_F1_SalesCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.max_wait END
           AS Home_FirstYear_F1_SalesCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.bought_by_phone, 0) = 1 AND COALESCE(ev.bought_online, 0) = 0) AND NOT (COALESCE(ev.bought_online, 0) = 1))) THEN NULL WHEN COALESCE(ev.q_online, 0) = 1 THEN 1 ELSE 0 END
           AS Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.bought_by_phone, 0) = 1 AND COALESCE(ev.bought_online, 0) = 0) AND NOT (COALESCE(ev.bought_online, 0) = 1))) THEN NULL WHEN (COALESCE(ev.q_online, 0) = 1) AND (COALESCE(ev.bought_by_phone, 0) = 1 AND COALESCE(ev.bought_online, 0) = 0) THEN 1 ELSE 0 END
           AS Home_FirstYear_F2_StartedOnlineFinishedByPhone_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.bought_by_phone, 0) = 1 AND COALESCE(ev.bought_online, 0) = 0) AND NOT (COALESCE(ev.bought_online, 0) = 1))) THEN NULL WHEN (COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.bought_by_phone, 0) = 1 AND COALESCE(ev.bought_online, 0) = 0) AND (COALESCE(ev.bought_online, 0) = 1) THEN 1 ELSE 0 END
           AS Home_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.no_price_online, 0) = 1) AND NOT (NOT (COALESCE(ev.no_price_online, 0) = 1)))) THEN NULL WHEN COALESCE(ev.q_online, 0) = 1 THEN 1 ELSE 0 END
           AS Home_FirstYear_F3_NoPriceOnline_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.no_price_online, 0) = 1) AND NOT (NOT (COALESCE(ev.no_price_online, 0) = 1)))) THEN NULL WHEN (COALESCE(ev.q_online, 0) = 1) AND (COALESCE(ev.no_price_online, 0) = 1) THEN 1 ELSE 0 END
           AS Home_FirstYear_F3_NoPriceOnline_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3}) OR ((COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.no_price_online, 0) = 1) AND NOT (NOT (COALESCE(ev.no_price_online, 0) = 1)))) THEN NULL WHEN (COALESCE(ev.q_online, 0) = 1) AND NOT (COALESCE(ev.no_price_online, 0) = 1) AND (NOT (COALESCE(ev.no_price_online, 0) = 1)) THEN 1 ELSE 0 END
           AS Home_FirstYear_F3_NoPriceOnline_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN ev.dispatch_first IS NOT NULL THEN 1 ELSE 0 END
           AS Home_FirstYear_F4_DocsChasedByCustomer_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN (ev.dispatch_first IS NOT NULL) AND (COALESCE(c2.chases, 0) > 0) THEN 1 ELSE 0 END
           AS Home_FirstYear_F4_DocsChasedByCustomer_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN (ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND (NOT (COALESCE(c2.chases, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_FirstYear_F4_DocsChasedByCustomer_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT ((ev.dispatch_first IS NOT NULL) AND (COALESCE(c2.chases, 0) > 0)) THEN NULL ELSE CASE WHEN c2.redispatched = 1 THEN 'Resolved' ELSE 'Unknown' END END
           AS Home_FirstYear_F4_DocsChasedByCustomer_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT (ev.dispatch_first IS NOT NULL) THEN NULL ELSE COALESCE(c2.chases, 0) END
           AS Home_FirstYear_F4_DocsChasedByCustomer_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4}) OR ((ev.dispatch_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT (ev.dispatch_first IS NOT NULL) THEN NULL ELSE datediff(c2.first_chase, ev.dispatch_first) END
           AS Home_FirstYear_F4_DocsChasedByCustomer_DaysFromDispatchToFirstChase,
       CAST(NULL AS INT)
           AS Home_FirstYear_F5_LoanSetupChase_Eligible,
       CAST(NULL AS INT)
           AS Home_FirstYear_F5_LoanSetupChase_Affected,
       CAST(NULL AS INT)
           AS Home_FirstYear_F5_LoanSetupChase_Comparison,
       CAST(NULL AS STRING)
           AS Home_FirstYear_F5_LoanSetupChase_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_R1_FailedLogin_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND (COALESCE(ev.logins_failed, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_R1_FailedLogin_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND (NOT (COALESCE(ev.logins_failed, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_Renewal_R1_FailedLogin_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND (COALESCE(ev.logins_failed, 0) > 0)) THEN NULL ELSE CASE WHEN ev.last_ok >= ev.first_failed THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_Renewal_R1_FailedLogin_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) THEN NULL ELSE COALESCE(ev.logins_failed, 0) END
           AS Home_Renewal_R1_FailedLogin_FailedLogins,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1}) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) THEN NULL ELSE COALESCE(ev.logins_ok, 0) END
           AS Home_Renewal_R1_FailedLogin_SuccessfulLogins,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(r2.n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_R2_RenewalCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_R2_RenewalCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND (NOT (COALESCE(r2.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_Renewal_R2_RenewalCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT ((COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN r2.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_Renewal_R2_RenewalCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n END
           AS Home_Renewal_R2_RenewalCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_conn END
           AS Home_Renewal_R2_RenewalCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_aband END
           AS Home_Renewal_R2_RenewalCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.max_wait END
           AS Home_Renewal_R2_RenewalCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN r3.first_login IS NOT NULL THEN 1 ELSE 0 END
           AS Home_Renewal_R3_OnlineThenCalled_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND (COALESCE(r3.called_after, 0) = 1) THEN 1 ELSE 0 END
           AS Home_Renewal_R3_OnlineThenCalled_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND (NOT (COALESCE(r3.called_after, 0) = 1)) THEN 1 ELSE 0 END
           AS Home_Renewal_R3_OnlineThenCalled_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) THEN NULL WHEN COALESCE(ev.requote_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_R4_RequoteNoPrice_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.requote_n, 0) > 0) AND (COALESCE(ev.rq_no_price, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_R4_RequoteNoPrice_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND (NOT (COALESCE(ev.rq_no_price, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_Renewal_R4_RequoteNoPrice_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) OR NOT ((COALESCE(ev.requote_n, 0) > 0) AND (COALESCE(ev.rq_no_price, 0) > 0)) THEN NULL ELSE CASE WHEN ev.last_price > ev.first_no_price THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_Renewal_R4_RequoteNoPrice_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) OR NOT (COALESCE(ev.requote_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rq_with_price, 0) END
           AS Home_Renewal_R4_RequoteNoPrice_RequotesWithPrice,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) OR NOT (COALESCE(ev.requote_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rq_no_price, 0) END
           AS Home_Renewal_R4_RequoteNoPrice_RequotesWithoutPrice,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR ((COALESCE(ev.requote_n, 0) > 0) AND NOT (COALESCE(ev.rq_no_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_no_price, 0) > 0)))) OR NOT (COALESCE(ev.requote_n, 0) > 0) THEN NULL ELSE CAST(NULL AS INT) END
           AS Home_Renewal_R4_RequoteNoPrice_RequotesAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5}) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN TRUE THEN 1 ELSE 0 END
           AS Home_Renewal_R5_RenewalHeld_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5}) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND (COALESCE(ev.held, 0) = 1) THEN 1 ELSE 0 END
           AS Home_Renewal_R5_RenewalHeld_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5}) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND (COALESCE(ev.invited, 0) = 1) THEN 1 ELSE 0 END
           AS Home_Renewal_R5_RenewalHeld_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5}) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) OR NOT ((TRUE) AND (COALESCE(ev.held, 0) = 1)) THEN NULL ELSE CASE WHEN ev.invited_last > ev.held_first THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_Renewal_R5_RenewalHeld_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R6}) OR ((TRUE) AND NOT (COALESCE(ev.date_missed, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN TRUE THEN 1 ELSE 0 END
           AS Home_Renewal_R6_RenewalDateMissed_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R6}) OR ((TRUE) AND NOT (COALESCE(ev.date_missed, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND (COALESCE(ev.date_missed, 0) = 1) THEN 1 ELSE 0 END
           AS Home_Renewal_R6_RenewalDateMissed_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R6}) OR ((TRUE) AND NOT (COALESCE(ev.date_missed, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND NOT (COALESCE(ev.date_missed, 0) = 1) AND (COALESCE(ev.invited, 0) = 1) THEN 1 ELSE 0 END
           AS Home_Renewal_R6_RenewalDateMissed_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(c1.n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Change_C1_ChangeCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c1.n, 0) > 0) AND (COALESCE(c1.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Change_C1_ChangeCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND (NOT (COALESCE(c1.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_Change_C1_ChangeCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT ((COALESCE(c1.n, 0) > 0) AND (COALESCE(c1.n_long, 0) > 0)) THEN NULL ELSE 'Unknown' END
           AS Home_Change_C1_ChangeCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n END
           AS Home_Change_C1_ChangeCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n_conn END
           AS Home_Change_C1_ChangeCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n_aband END
           AS Home_Change_C1_ChangeCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.max_wait END
           AS Home_Change_C1_ChangeCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_DocRequest_C3_DocRequestCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Home_DocRequest_C3_DocRequestCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND (NOT (COALESCE(c3c.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Home_DocRequest_C3_DocRequestCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN c3c.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Home_DocRequest_C3_DocRequestCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n END
           AS Home_DocRequest_C3_DocRequestCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_conn END
           AS Home_DocRequest_C3_DocRequestCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_aband END
           AS Home_DocRequest_C3_DocRequestCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.max_wait END
           AS Home_DocRequest_C3_DocRequestCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_DocRequest_C4_DocNotArrived_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1) THEN 1 ELSE 0 END
           AS Home_DocRequest_C4_DocNotArrived_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND (NOT (COALESCE(c3c.n, 0) > 1)) THEN 1 ELSE 0 END
           AS Home_DocRequest_C4_DocNotArrived_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1)) THEN NULL ELSE 'Unknown' END
           AS Home_DocRequest_C4_DocNotArrived_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE COALESCE(c3c.n, 0) - 1 END
           AS Home_DocRequest_C4_DocNotArrived_EventCount,
       CAST(NULL AS INT)
           AS Home_DocRequest_C5_DocFromInsurer_Eligible,
       CAST(NULL AS INT)
           AS Home_DocRequest_C5_DocFromInsurer_Affected,
       CAST(NULL AS INT)
           AS Home_DocRequest_C5_DocFromInsurer_Comparison,
       CAST(NULL AS STRING)
           AS Home_DocRequest_C5_DocFromInsurer_Resolution,
       CAST(NULL AS INT)
           AS Home_FirstYear_HA5B1_LoanSetup_Eligible,
       CAST(NULL AS INT)
           AS Home_FirstYear_HA5B1_LoanSetup_Entered,
       CAST(NULL AS INT)
           AS Home_FirstYear_HA5B1_LoanSetup_Comparison,
       CAST(NULL AS INT)
           AS Home_FirstYear_HA5B1_LoanSetup_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Home_FirstYear_HA6B1_DocIntakeAfterSale_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) AND COALESCE(ev.fy_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_FirstYear_HA6B1_DocIntakeAfterSale_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) AND NOT COALESCE(ev.fy_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_FirstYear_HA6B1_DocIntakeAfterSale_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) OR NOT ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.fy_doc_n, 0) END
           AS Home_FirstYear_HA6B1_DocIntakeAfterSale_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_HR2B1_OnlineRequote_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) AND COALESCE(ev.requote_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR2B1_OnlineRequote_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) AND NOT COALESCE(ev.requote_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR2B1_OnlineRequote_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) OR NOT ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) THEN NULL ELSE COALESCE(ev.requote_win_n, 0) END
           AS Home_Renewal_HR2B1_OnlineRequote_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.rn_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_HR4B2_DocIntakeAfterRenewal_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.rn_doc_n, 0) > 0) AND COALESCE(ev.rn_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR4B2_DocIntakeAfterRenewal_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.rn_doc_n, 0) > 0) AND NOT COALESCE(ev.rn_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR4B2_DocIntakeAfterRenewal_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) OR NOT ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.rn_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rn_doc_n, 0) END
           AS Home_Renewal_HR4B2_DocIntakeAfterRenewal_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Monthly}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.monthly_n, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Renewal_HR4cB1_NewMonthlyAgreement_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Monthly}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.monthly_n, 0) > 0) AND COALESCE(ev.monthly_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR4cB1_NewMonthlyAgreement_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Monthly}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.monthly_n, 0) > 0) AND NOT COALESCE(ev.monthly_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Renewal_HR4cB1_NewMonthlyAgreement_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Monthly}) OR e.is_open = 1) OR NOT ((COALESCE(ev.paid, 0) = 1) OR COALESCE(ev.monthly_n, 0) > 0) THEN NULL ELSE COALESCE(ev.monthly_n, 0) END
           AS Home_Renewal_HR4cB1_NewMonthlyAgreement_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Home_Change_HM4B1_DocIntakeForChange_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) AND COALESCE(ev.change_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Change_HM4B1_DocIntakeForChange_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) AND NOT COALESCE(ev.change_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Home_Change_HM4B1_DocIntakeForChange_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) OR NOT ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.change_doc_n, 0) END
           AS Home_Change_HM4B1_DocIntakeForChange_EventCount
FROM   stg.fe3m_home_ep e
LEFT JOIN stg.fe3m_home_evagg ev ON ev.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_home_ev2   c2 ON c2.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_home_r3    r3 ON r3.EpisodeID = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_home_callagg WHERE m = 'F1') f1  ON f1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_home_callagg WHERE m = 'R2') r2  ON r2.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_home_callagg WHERE m = 'C1') c1  ON c1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_home_callagg WHERE m = 'C3') c3c ON c3c.EpisodeID = e.EpisodeID
;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Checks

In [10]:
-- Grain: one row per Home episode (expect duplicate_ids = 0).
SELECT (SELECT COUNT(*) FROM stg.fe3m_home) AS rows_, (SELECT COUNT(*) FROM stg.fe3m_home_ep) AS episodes,
       (SELECT COUNT(*) - COUNT(DISTINCT EpisodeID) FROM stg.fe3m_home) AS duplicate_ids;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [11]:
-- Per measure: rows filled, eligible, numerator, comparison, invariant violations (expect bad_groups = 0).
SELECT 'Home_FirstYear_F1_SalesCallWait' AS measure, COUNT(Home_FirstYear_F1_SalesCallWait_Eligible) AS rows_filled, SUM(Home_FirstYear_F1_SalesCallWait_Eligible) AS eligible, SUM(Home_FirstYear_F1_SalesCallWait_Affected) AS numerator,
       SUM(Home_FirstYear_F1_SalesCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_F1_SalesCallWait_Eligible = 1 AND COALESCE(Home_FirstYear_F1_SalesCallWait_Affected, 0) + COALESCE(Home_FirstYear_F1_SalesCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_F2_StartedOnlineFinishedByPhone' AS measure, COUNT(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS rows_filled, SUM(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS eligible, SUM(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Affected) AS numerator,
       SUM(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 AND COALESCE(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Affected, 0) + COALESCE(Home_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_F3_NoPriceOnline' AS measure, COUNT(Home_FirstYear_F3_NoPriceOnline_Eligible) AS rows_filled, SUM(Home_FirstYear_F3_NoPriceOnline_Eligible) AS eligible, SUM(Home_FirstYear_F3_NoPriceOnline_Affected) AS numerator,
       SUM(Home_FirstYear_F3_NoPriceOnline_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_F3_NoPriceOnline_Eligible = 1 AND COALESCE(Home_FirstYear_F3_NoPriceOnline_Affected, 0) + COALESCE(Home_FirstYear_F3_NoPriceOnline_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_F4_DocsChasedByCustomer' AS measure, COUNT(Home_FirstYear_F4_DocsChasedByCustomer_Eligible) AS rows_filled, SUM(Home_FirstYear_F4_DocsChasedByCustomer_Eligible) AS eligible, SUM(Home_FirstYear_F4_DocsChasedByCustomer_Affected) AS numerator,
       SUM(Home_FirstYear_F4_DocsChasedByCustomer_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 AND COALESCE(Home_FirstYear_F4_DocsChasedByCustomer_Affected, 0) + COALESCE(Home_FirstYear_F4_DocsChasedByCustomer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_F5_LoanSetupChase' AS measure, COUNT(Home_FirstYear_F5_LoanSetupChase_Eligible) AS rows_filled, SUM(Home_FirstYear_F5_LoanSetupChase_Eligible) AS eligible, SUM(Home_FirstYear_F5_LoanSetupChase_Affected) AS numerator,
       SUM(Home_FirstYear_F5_LoanSetupChase_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_F5_LoanSetupChase_Eligible = 1 AND COALESCE(Home_FirstYear_F5_LoanSetupChase_Affected, 0) + COALESCE(Home_FirstYear_F5_LoanSetupChase_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R1_FailedLogin' AS measure, COUNT(Home_Renewal_R1_FailedLogin_Eligible) AS rows_filled, SUM(Home_Renewal_R1_FailedLogin_Eligible) AS eligible, SUM(Home_Renewal_R1_FailedLogin_Affected) AS numerator,
       SUM(Home_Renewal_R1_FailedLogin_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R1_FailedLogin_Eligible = 1 AND COALESCE(Home_Renewal_R1_FailedLogin_Affected, 0) + COALESCE(Home_Renewal_R1_FailedLogin_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R2_RenewalCallWait' AS measure, COUNT(Home_Renewal_R2_RenewalCallWait_Eligible) AS rows_filled, SUM(Home_Renewal_R2_RenewalCallWait_Eligible) AS eligible, SUM(Home_Renewal_R2_RenewalCallWait_Affected) AS numerator,
       SUM(Home_Renewal_R2_RenewalCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R2_RenewalCallWait_Eligible = 1 AND COALESCE(Home_Renewal_R2_RenewalCallWait_Affected, 0) + COALESCE(Home_Renewal_R2_RenewalCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R3_OnlineThenCalled' AS measure, COUNT(Home_Renewal_R3_OnlineThenCalled_Eligible) AS rows_filled, SUM(Home_Renewal_R3_OnlineThenCalled_Eligible) AS eligible, SUM(Home_Renewal_R3_OnlineThenCalled_Affected) AS numerator,
       SUM(Home_Renewal_R3_OnlineThenCalled_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R3_OnlineThenCalled_Eligible = 1 AND COALESCE(Home_Renewal_R3_OnlineThenCalled_Affected, 0) + COALESCE(Home_Renewal_R3_OnlineThenCalled_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R4_RequoteNoPrice' AS measure, COUNT(Home_Renewal_R4_RequoteNoPrice_Eligible) AS rows_filled, SUM(Home_Renewal_R4_RequoteNoPrice_Eligible) AS eligible, SUM(Home_Renewal_R4_RequoteNoPrice_Affected) AS numerator,
       SUM(Home_Renewal_R4_RequoteNoPrice_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R4_RequoteNoPrice_Eligible = 1 AND COALESCE(Home_Renewal_R4_RequoteNoPrice_Affected, 0) + COALESCE(Home_Renewal_R4_RequoteNoPrice_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R5_RenewalHeld' AS measure, COUNT(Home_Renewal_R5_RenewalHeld_Eligible) AS rows_filled, SUM(Home_Renewal_R5_RenewalHeld_Eligible) AS eligible, SUM(Home_Renewal_R5_RenewalHeld_Affected) AS numerator,
       SUM(Home_Renewal_R5_RenewalHeld_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R5_RenewalHeld_Eligible = 1 AND COALESCE(Home_Renewal_R5_RenewalHeld_Affected, 0) + COALESCE(Home_Renewal_R5_RenewalHeld_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_R6_RenewalDateMissed' AS measure, COUNT(Home_Renewal_R6_RenewalDateMissed_Eligible) AS rows_filled, SUM(Home_Renewal_R6_RenewalDateMissed_Eligible) AS eligible, SUM(Home_Renewal_R6_RenewalDateMissed_Affected) AS numerator,
       SUM(Home_Renewal_R6_RenewalDateMissed_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_R6_RenewalDateMissed_Eligible = 1 AND COALESCE(Home_Renewal_R6_RenewalDateMissed_Affected, 0) + COALESCE(Home_Renewal_R6_RenewalDateMissed_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Change_C1_ChangeCallWait' AS measure, COUNT(Home_Change_C1_ChangeCallWait_Eligible) AS rows_filled, SUM(Home_Change_C1_ChangeCallWait_Eligible) AS eligible, SUM(Home_Change_C1_ChangeCallWait_Affected) AS numerator,
       SUM(Home_Change_C1_ChangeCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Home_Change_C1_ChangeCallWait_Eligible = 1 AND COALESCE(Home_Change_C1_ChangeCallWait_Affected, 0) + COALESCE(Home_Change_C1_ChangeCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_DocRequest_C3_DocRequestCallWait' AS measure, COUNT(Home_DocRequest_C3_DocRequestCallWait_Eligible) AS rows_filled, SUM(Home_DocRequest_C3_DocRequestCallWait_Eligible) AS eligible, SUM(Home_DocRequest_C3_DocRequestCallWait_Affected) AS numerator,
       SUM(Home_DocRequest_C3_DocRequestCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Home_DocRequest_C3_DocRequestCallWait_Eligible = 1 AND COALESCE(Home_DocRequest_C3_DocRequestCallWait_Affected, 0) + COALESCE(Home_DocRequest_C3_DocRequestCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_DocRequest_C4_DocNotArrived' AS measure, COUNT(Home_DocRequest_C4_DocNotArrived_Eligible) AS rows_filled, SUM(Home_DocRequest_C4_DocNotArrived_Eligible) AS eligible, SUM(Home_DocRequest_C4_DocNotArrived_Affected) AS numerator,
       SUM(Home_DocRequest_C4_DocNotArrived_Comparison) AS comparison,
       SUM(CASE WHEN Home_DocRequest_C4_DocNotArrived_Eligible = 1 AND COALESCE(Home_DocRequest_C4_DocNotArrived_Affected, 0) + COALESCE(Home_DocRequest_C4_DocNotArrived_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_DocRequest_C5_DocFromInsurer' AS measure, COUNT(Home_DocRequest_C5_DocFromInsurer_Eligible) AS rows_filled, SUM(Home_DocRequest_C5_DocFromInsurer_Eligible) AS eligible, SUM(Home_DocRequest_C5_DocFromInsurer_Affected) AS numerator,
       SUM(Home_DocRequest_C5_DocFromInsurer_Comparison) AS comparison,
       SUM(CASE WHEN Home_DocRequest_C5_DocFromInsurer_Eligible = 1 AND COALESCE(Home_DocRequest_C5_DocFromInsurer_Affected, 0) + COALESCE(Home_DocRequest_C5_DocFromInsurer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_HA5B1_LoanSetup' AS measure, COUNT(Home_FirstYear_HA5B1_LoanSetup_Eligible) AS rows_filled, SUM(Home_FirstYear_HA5B1_LoanSetup_Eligible) AS eligible, SUM(Home_FirstYear_HA5B1_LoanSetup_Entered) AS numerator,
       SUM(Home_FirstYear_HA5B1_LoanSetup_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_HA5B1_LoanSetup_Eligible = 1 AND COALESCE(Home_FirstYear_HA5B1_LoanSetup_Entered, 0) + COALESCE(Home_FirstYear_HA5B1_LoanSetup_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_FirstYear_HA6B1_DocIntakeAfterSale' AS measure, COUNT(Home_FirstYear_HA6B1_DocIntakeAfterSale_Eligible) AS rows_filled, SUM(Home_FirstYear_HA6B1_DocIntakeAfterSale_Eligible) AS eligible, SUM(Home_FirstYear_HA6B1_DocIntakeAfterSale_Entered) AS numerator,
       SUM(Home_FirstYear_HA6B1_DocIntakeAfterSale_Comparison) AS comparison,
       SUM(CASE WHEN Home_FirstYear_HA6B1_DocIntakeAfterSale_Eligible = 1 AND COALESCE(Home_FirstYear_HA6B1_DocIntakeAfterSale_Entered, 0) + COALESCE(Home_FirstYear_HA6B1_DocIntakeAfterSale_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_HR2B1_OnlineRequote' AS measure, COUNT(Home_Renewal_HR2B1_OnlineRequote_Eligible) AS rows_filled, SUM(Home_Renewal_HR2B1_OnlineRequote_Eligible) AS eligible, SUM(Home_Renewal_HR2B1_OnlineRequote_Entered) AS numerator,
       SUM(Home_Renewal_HR2B1_OnlineRequote_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_HR2B1_OnlineRequote_Eligible = 1 AND COALESCE(Home_Renewal_HR2B1_OnlineRequote_Entered, 0) + COALESCE(Home_Renewal_HR2B1_OnlineRequote_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_HR4B2_DocIntakeAfterRenewal' AS measure, COUNT(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Eligible) AS rows_filled, SUM(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Eligible) AS eligible, SUM(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Entered) AS numerator,
       SUM(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_HR4B2_DocIntakeAfterRenewal_Eligible = 1 AND COALESCE(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Entered, 0) + COALESCE(Home_Renewal_HR4B2_DocIntakeAfterRenewal_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Renewal_HR4cB1_NewMonthlyAgreement' AS measure, COUNT(Home_Renewal_HR4cB1_NewMonthlyAgreement_Eligible) AS rows_filled, SUM(Home_Renewal_HR4cB1_NewMonthlyAgreement_Eligible) AS eligible, SUM(Home_Renewal_HR4cB1_NewMonthlyAgreement_Entered) AS numerator,
       SUM(Home_Renewal_HR4cB1_NewMonthlyAgreement_Comparison) AS comparison,
       SUM(CASE WHEN Home_Renewal_HR4cB1_NewMonthlyAgreement_Eligible = 1 AND COALESCE(Home_Renewal_HR4cB1_NewMonthlyAgreement_Entered, 0) + COALESCE(Home_Renewal_HR4cB1_NewMonthlyAgreement_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home
UNION ALL
SELECT 'Home_Change_HM4B1_DocIntakeForChange' AS measure, COUNT(Home_Change_HM4B1_DocIntakeForChange_Eligible) AS rows_filled, SUM(Home_Change_HM4B1_DocIntakeForChange_Eligible) AS eligible, SUM(Home_Change_HM4B1_DocIntakeForChange_Entered) AS numerator,
       SUM(Home_Change_HM4B1_DocIntakeForChange_Comparison) AS comparison,
       SUM(CASE WHEN Home_Change_HM4B1_DocIntakeForChange_Eligible = 1 AND COALESCE(Home_Change_HM4B1_DocIntakeForChange_Entered, 0) + COALESCE(Home_Change_HM4B1_DocIntakeForChange_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_home;

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 26, Finished, Available, Finished, False)

<Spark SQL result set with 21 rows and 6 fields>

In [12]:
%%pyspark
mssparkutils.session.stop()

StatementMeta(, a1187c34-8bbd-4724-a04e-eab814cea29c, 28, Finished, Available, Finished, False)